<a href="https://colab.research.google.com/github/mikaellycardoso/EcoAtlas/blob/main/Avalia%C3%A7%C3%A3o_C2_A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação C2.A1 - Análise de Dados
**Grupo**:

Anna Luiza Tamanini Andrade

Laisa de Souza Camilo

Mikaelly do Nascimento Cardoso

Victória Sofia dos Santos Teixeira

**Disciplina:**
Análise de Dados aplicada a computação.


**Objetivo:** Análise de dados de empregabilidade e predição de salários e status de recolocação.

## **Parte 1: Setup do Ambiente e e Pré-processamento**

Item: 1, 2, 3 e 5

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

%matplotlib inline
sns.set(style="whitegrid")

df_original = pd.read_csv('Placement_Data_Full_Class.csv')

df_no_salary = df_original.copy().drop(columns=['salary'])

df_original.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Placement_Data_Full_Class.csv'

## **Parte 2: Codificação e Preparação de Dados para Modelagem**


## 2. Pesquisa: Técnica de Codificação LabelEncoder

### O que é o LabelEncoder?
O **LabelEncoder** é uma técnica de pré-processamento de dados utilizada para transformar variáveis categóricas (dados em formato de texto) em valores numéricos inteiros.


### Como funciona?
O algoritmo atribui um número único para cada categoria presente em uma coluna, geralmente seguindo a ordem alfabética.
* **Exemplo:** Na coluna `gender`, o rótulo "Female" pode ser convertido em `0` e "Male" em `1`.

### Por que utilizar neste projeto?
A maioria dos modelos de *Machine Learning* (como a **Regressão Logística** e a **Regressão Linear** que utilizaremos adiante) baseia-se em cálculos matemáticos e equações matriciais. Estes algoritmos não conseguem processar palavras diretamente. Portanto, a codificação é um passo obrigatório para converter as categorias em um formato que o computador consiga interpretar matematicamente.

### Aplicação nos Dados
Neste conjunto de dados, o LabelEncoder será aplicado para converter os seguintes atributos:
* **Atributos Categóricos:** `gender`, `ssc_b`, `hsc_b`, `hsc_s`, `degree_t`, `workex` e `specialisation`.
* **Variável Alvo (Target):** `status` (convertendo "Placed" e "Not Placed" em valores binários).

### **7. Aplicando a Codificação LabelEncoder no DataFrame `df_no_salary`**

In [ ]:
from sklearn.preprocessing import LabelEncoder

codificador_rotulos = LabelEncoder()

colunas_para_codificar = ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation', 'status']

for coluna in colunas_para_codificar:
    df_no_salary[coluna] = codificador_rotulos.fit_transform(df_no_salary[coluna])

print("Categorias convertidas em df_no_salary.")
df_no_salary.head()

###**10. Preparando o `df_no_status` e Separando Dados para Predição de Salários**

In [ ]:
df_no_status = df_original.drop(columns=['status']).copy()

cols_no_status = ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation']

for col in cols_no_status:
    df_no_status[col] = codificador_rotulos.fit_transform(df_no_status[col])

df_missing_salary = df_no_status[df_no_status['salary'].isnull()]

df_with_salary = df_no_status[df_no_status['salary'].notnull()]


print(f"Total de registros sem salário para predição: {len(df_missing_salary)}")

print("\nDataFrame sem status (com colunas categóricas codificadas e sem 'status'):")
display(df_no_status.head())

print("\nDataFrame com salário ausente (registros para predição de salário):")
display(df_missing_salary.head())

print("\nDataFrame com salário (registros para treino do modelo de predição de salário):")
display(df_with_salary.head())

#6. Geração de Gráficos (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#1. Carregando os dados
df_original = pd.read_csv('Placement_Data_Full_Class.csv')

#2. Configurando a área dos gráficos
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

#Gráfico 1: Distribuição de Status
sns.countplot(data=df_original, x='status', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Distribuição de Status (Contratados vs Não Contratados)')

#Gráfico 2: Notas de Graduação vs Status
sns.boxplot(data=df_original, x='status', y='degree_p', ax=axes[0, 1], color='lightsteelblue')
axes[0, 1].set_title('Impacto das Notas de Graduação no Status')

#Gráfico 3: Experiência de Trabalho vs Status
sns.countplot(data=df_original, x='workex', hue='status', ax=axes[1, 0])
axes[1, 0].set_title('Relação entre Experiência Prévia e Contratação')

#Gráfico 4: Distribuição Salarial
sns.histplot(df_original[df_original['status'] == 'Placed']['salary'], kde=True, color='teal', ax=axes[1, 1])
axes[1, 1].set_title('Distribuição Salarial dos Candidatos Contratados')

plt.tight_layout()
plt.show()

#Conclusões sobre o comportamento dos dados:

**1. Perfil de Contratação**
O mercado mostrou-se positivo: a maioria dos alunos (68%) conseguiu uma vaga. No entanto, o processo é seletivo, já que 67 alunos não foram contratados, mostrando que o sucesso não é garantido para todos

**2. O Peso das Notas**
Notei que quem foi contratado manteve médias escolares bem mais altas, geralmente acima de 70%, desde o ensino fundamental até à faculdade. Alunos com notas baixas tiveram muito mais dificuldade, confirmando que o histórico escolar é um filtro real

**3. Valor da Experiência**
 Ter trabalhado antes faz uma diferença enorme. Mais de 86% de quem tinha experiência foi contratado, enquanto para quem nunca trabalhou as chances foram bem menores

 **4. Curso e Salário**
Escolha do curso e ganhos: Alunos de Marketing e Finanças saíram na frente em vagas e salários. Curiosamente, a nota no teste de aptidão (etest_p) não influenciou tanto, provando que o currículo e a prática valem mais que um teste isolado.

**Organização:** O dataset está consistente, com salários preenchidos apenas para quem foi contratado, o que confirma a integridade da nossa análise.